# BPI Challenge 2012: loan applications, offers and decision quality

The question is whether a lender can reach a clear decision sooner while preserving required checks and the applicant’s choice. A fast decline, a cancelled application and an activated loan are different outcomes; none should be assigned a poor-performance label just because it is a negative commercial outcome.

The analysis follows [PM²](https://vdaalst.com/publications/p824.pdf): Planning, Extraction, Data Processing, Mining and Analysis, Evaluation, and Process Improvement and Support. Each stage leaves a journal entry and an evidence question. Scores, priorities and sensitivity calculations come from WISE; this notebook selects questions and presents their results. No numerical finding is assumed before the source log has been analysed.

The source is an actual loan-application event log. The native application, offer and work labels are retained so that a work lifecycle observation is not mistaken for a new offer or application. [Dataset catalogue](https://research.tue.nl/en/datasets/bpi-challenge-2012/).

**Planning question.** Separate internal assessment delay, applicant response, repeated offer creation and incomplete observation. Review the first submission-to-preacceptance interval alongside the first terminal application outcome, then inspect whether high-priority cases reflect avoidable document chasing or a necessary credit decision.

In [ ]:
import os
from dataclasses import replace
from pathlib import Path
from IPython.display import display, Markdown
import pandas as pd
import matplotlib.pyplot as plt
import wise

if wise.__version__ != "1.0.1":
    raise RuntimeError("Select the environment with the frozen 1.0.1 package.")

from bpic_analysis import (
    challenge, summary, norm_table, criterion_summary, activity_summary,
    context_profiles, priorities, plot_overview, case_evidence,
    threshold_sensitivity, view_agreement, integrity, journal, save_outputs, explanations,
)

DATA_ROOT = os.environ.get("WISE_PAPER_DATA")
EXPORT_RESULTS = False

## 1. Extraction: identify the population and observation boundary

Read each source from the supplied data folder. The summary records actual source populations and assessment status. Retain every observation of selected cases, including records outside a selected start-month cohort. A context extract without an event sequence is identified separately; it is not assigned a process score.

In [ ]:
runs = challenge(2012, data_folder=DATA_ROOT)
display(summary(runs))
scored_runs = {name: run for name, run in runs.items() if run.get("result") is not None}
display(journal(runs))

## 2. Data processing: check the meaning of observed activities

Application states (`A_...`), offer states (`O_...`) and work activities (`W_...`) answer different questions. A work activity can be scheduled, started and completed; counting all lifecycle records as separate customer contacts would overstate work. Multiple `O_CREATED` observations can represent genuine customer choice. The first `O_SENT` to first `O_SENT_BACK` at or after it interval mixes customer response with administration and cannot be labelled staff delay.

The tables below distinguish event occurrences from cases with an observation. Counts are descriptive evidence; they are not effort, root causes or success rates.

In [ ]:
ACTIVITY_ANCHORS = ['A_SUBMITTED', 'A_PREACCEPTED', 'A_ACCEPTED', 'A_APPROVED', 'A_ACTIVATED', 'A_DECLINED', 'A_CANCELLED', 'O_CREATED', 'O_SENT', 'O_SENT_BACK', 'W_Nabellen incomplete dossiers', 'W_Nabellen offertes']
for name, run in scored_runs.items():
    display(Markdown(f"### {name}"))
    display(activity_summary(run, labels=ACTIVITY_ANCHORS))
    display(activity_summary(run).head(25))

## 3. Translate the business goal into a declared norm

The exploratory norm asks about prompt initial assessment, timely first application outcome, offer response and offer-creation intensity. Its day cut points are diagnostic scenarios, not contractual service levels. The offer-count criterion identifies cases to inspect; it does not say that offering alternatives is wrong. Approval before every activation and counts of completed follow-up work require additional lifecycle or ordering rules; keep those questions outside the scored native norm until their semantics are validated.

Read the criterion, scope, thresholds and view weights before interpreting a score. For `Singularity`, `k` is the allowed count and `K` is the additional count range over which severity increases; `K` is not an upper allowed count. Native lag matches the first response at or after the first activation, allowing equal timestamps. Pending questions are shown so that an unsupported business requirement cannot disappear behind a convincing chart.

In [ ]:
for name, run in scored_runs.items():
    display(Markdown(f"### {name}"))
    display(norm_table(run))
    display(run["specification"].get("pending_checks", []))

## 4. Establish the denominator and evidence coverage

An application with no recorded terminal outcome is not a successful fast decision. Missing lag endpoints remain unavailable. Check whether the timing evidence is concentrated in completed applications while the unanswered or cancelled cases disappear from the ranking. Coverage and in-scope counts must accompany the severity chart.

`evaluated_cases` and `in_scope_cases` show different denominators. Mean severity refers to usable evidence; an unavailable observation is not a zero violation. A high mean severity with narrow coverage deserves a different response from a widespread, well-observed problem.

The library renormalises the score over usable evidence. A high score cannot certify adherence to the full norm when some evidence is unavailable. Criterion coverage is the unweighted fraction of in-scope cases that can be evaluated for that criterion. The case witness also shows each view’s weighted evidence coverage. These are different measures, and neither certifies that the source contains every real process event.

In [ ]:
criterion_tables = {}
for name, run in scored_runs.items():
    criterion_tables[name] = criterion_summary(run)
    display(Markdown(f"### {name}"))
    display(criterion_tables[name])

## 5. Mining and analysis: compare stakeholder priorities

The timeliness view tests where elapsed intervals deserve attention; the control-and-quality view puts more weight on the supported rework signals. Read the actual norm table: a view name does not mean that its pending control checks are implemented. A different top slice is a difference in investigation preference, not a contradiction in the source data.

Priority combines scored case volume with a deficit in conformance score relative to the scored reference population. A higher conformance score means closer adherence to the declared norm; higher criterion severity means more deviation. The first rows show where to investigate, not which employee or process is at fault. An unranked or below-reference slice can still contain important individual cases.

In [ ]:
view_backlogs = {}
for name, run in scored_runs.items():
    for view in run["result"].views:
        backlog = priorities(run, view=view, gamma=0)
        view_backlogs[(name, view)] = backlog
        display(Markdown(f"### {name} — {view}"))
        display(backlog.table.head(10))

## 6. Visualise severity, evidence and investigation order

The overview uses computed library results. Read the severity and coverage panels together, then compare their message with the priority list. A frequent activity is not automatically an important failure; a rare signal should retain access to its supporting case.

In [ ]:
for name, run in scored_runs.items():
    display(Markdown(f"### {name}"))
    for figure in plot_overview(run):
        display(figure)
        plt.close(figure)
    table = criterion_tables[name]
    if not table.empty:
        figure, axis = plt.subplots(figsize=(8, 5), constrained_layout=True)
        plotted = table.dropna(subset=["evidence_coverage", "mean_severity"])
        for row in plotted.itertuples(index=False):
            axis.scatter(row.evidence_coverage, row.mean_severity, s=85,
                         label=row.criterion_id)
        axis.legend(title="Criterion", loc="center left", bbox_to_anchor=(1, 0.5))
        axis.set(xlim=(-0.03, 1.08), ylim=(-0.03, 1.08),
                 xlabel="Evidence coverage within business scope",
                 ylabel="Mean severity within usable evidence",
                 title="Does a severe signal also have broad usable evidence?")
        display(figure)
        plt.close(figure)

## 7. Explain the signed layer drivers

A positive timing contribution can be offset by below-reference offer rework. Preserve both signs before judging a segment. Where a slice is driven mostly by the offer-response criterion, first inspect who was expected to act and whether the applicant had enough information; do not assign the full elapsed interval to an internal assessor.

Positive contributions increase the aggregate gap; negative contributions offset it. WISE keeps the signs and clips only the final sum. Do not remove the offsets to make a problem look larger.

In [ ]:
for (name, view), backlog in view_backlogs.items():
    explanation = explanations(scored_runs[name], backlog)
    display(Markdown(f"### {name} — {view}"))
    display(explanation["contributions"].reindex(backlog.table.head(10).index))
    display(explanation["drivers"].reindex(backlog.table.head(10).index))

## 8. Test thresholds, preferences and small-slice sensitivity

Compare half, baseline and double timing thresholds and severity widths and ask whether the same applications or case cohorts remain worth investigating. If a lead appears only under one narrow setting, treat it as threshold-dependent. Compare small-slice shrinkage as well: a rare loan amount is not automatically a recurring operational problem.

These are declared scenarios, not estimates of an optimal policy. The timing scenario scales both the allowed lag and its severity width. The occurrence scenario keeps the allowed count `k` fixed and scales the additional severity width `K`; it does not halve or double the allowed count. The overlap measure concerns positive-priority shortlist membership and is undefined when both lists are empty. Low agreement is a prompt to explain the business trade-off.

In [ ]:
for name, run in scored_runs.items():
    display(Markdown(f"### {name}"))
    display(threshold_sensitivity(run, factors=(0.5, 1, 2)))
    display(view_agreement(run, k=10))
    view = run["result"].views[0]
    reference = priorities(run, view=view, gamma=0)
    for gamma in (5, 20, 100):
        adjusted = priorities(run, view=view, gamma=gamma)
        display({"dataset": name, "view": view, "gamma": gamma,
                 "positive_top10_overlap": wise.top_k_overlap(reference, adjusted, k=10)})
        display(adjusted.table.head(10))

## 9. Compare complete cases in meaningful context

Requested amount is whole-application context, not an amount attached only to the event where it appears. Raw amount slices may be small or numerous and should be read beside their case counts. Compare case-start cohorts without discarding their later events. An observed span is the time between first and last recorded events; it is not necessarily decision time or verified completion.

A start month selects cases rather than splitting their events. Derived first/last observed months use UTC after timestamp conversion; they are not verified business-local calendar months. A business-calendar analysis needs the source timezone and its calendar rules. `last_observed_month` and `observed_span_band` describe observation boundaries. They are not verified finish-month or completion measures. A context field that changes within a case is not silently treated as a stable business attribute in the comparisons below.

In [ ]:
CONTEXT_KEYS = ['AMOUNT_REQ', 'case_start_month', 'observed_span_band']
for name, run in scored_runs.items():
    display(Markdown(f"### {name}"))
    display(context_profiles(run, attributes=CONTEXT_KEYS))
    varying = run["log"].varying_case_attributes
    for attribute in CONTEXT_KEYS:
        if attribute not in run["result"].cases.columns:
            continue
        if varying.get(attribute, 0):
            display({"attribute": attribute, "comparison": "Context changes within cases; review the classification rule first"})
            continue
        display(Markdown(f"#### Whole-case grouping: {attribute}"))
        display(priorities(run, by=attribute, gamma=20).table.head(10))

## 10. Test how missing evidence changes the aggregation

Flat and layer-balanced scoring answer slightly different aggregation questions when evidence availability varies. Rescore the same complete cases through the library. Compare the resulting investigation order using identical slice keys and view; do not interpret a change in order as a newly observed process event.

In [ ]:
for name, run in scored_runs.items():
    view = run["result"].views[0]
    reference = priorities(run, view=view, gamma=0)
    by = list(reference.spec.by)
    for mode in ("flat", "layer_balanced"):
        rescored = replace(wise.score(run["log"], run["norm"], mode=mode), cases=run["result"].cases)
        backlog = wise.prioritize(rescored, by=by, view=view, gamma=0)
        display({"dataset": name, "view": view, "mode": mode,
                 "positive_top10_overlap": wise.top_k_overlap(reference, backlog, k=10)})
        display(backlog.table.head(10))

## 11. Evaluation: inspect the full case behind a lead

Follow a case highlighted for repeated offer creation from submission through every offer and work observation. Check whether the extra offers concern changes requested by the applicant, missing documents or correction of an internal error. Then inspect a timing criterion in the same complete case. A case selected by a count alone cannot establish which mechanism caused the delay.

The witness retains its source sequence and whole-case context. Compare the library evaluation with the matching observations and note what a missing endpoint means. The default witness is an investigation example, not a representative sample or proof of causation. To examine another case, pass its identifier to `case_evidence` without filtering away other events.

In [ ]:
case_witnesses = {}
for name, run in scored_runs.items():
    witness = case_evidence(run)
    case_witnesses[name] = witness
    display(Markdown(f"### Full source case: {name}"))
    for label, value in witness.items():
        display(Markdown(f"**{label}**"))
        display(value)
offer_witness = case_evidence(scored_runs["bpic2012"], criterion="L12_C04")
for label, value in offer_witness.items():
    display(Markdown(f"**Offer-creation witness: {label}**"))
    display(value)

### Place the case evidence on a timeline

This figure uses event timestamps returned by the library for a complete recorded case selected by the first diagnostic criterion. Named lanes highlight norm activities; the grey lane keeps the remaining recorded activities visible. Coincident points can overlap and establish no order. Undated events remain in the source table and cannot be placed on the time axis. The selected case is an investigation example, not a representative sample; return to its full source witness before interpreting an apparent gap.

In [ ]:
from bpic_analysis import plot_case_timeline
for identifier, run in runs.items():
    if "result" not in run:
        continue
    figure = plot_case_timeline(run)
    display(figure)
    plt.close(figure)

## 12. Validate the result and decide what it supports

A lead supports a concrete process hypothesis only when the source sequence and ownership information agree with it. If multiple follow-ups repeat the same missing-document request, a consolidated checklist may help. If offer changes express applicant choice or necessary risk checks, reducing the count would damage the process. Record which explanation the witness supports and what information remains missing.

The integrity check records norm round-trip consistency, score decomposition, assessed case-index agreement and the retained event count. It does not independently certify extraction completeness, historical field validity or coverage of the real process. A failed check or incomplete binding limits the conclusion even when a plot looks convincing. Keep unsupported checks and cases without usable evidence in the journal.

### Baseline findings to carry into case review

The bound extract contains 13,087 applications and 262,200 events. The baseline norm yields the following leads; the tables above remain the authoritative results if the input or norm changes.

- The first recorded submission-to-terminal-outcome interval is usable for 12,688 applications; 399 have no matched response. Its mean severity is 0.206 under the diagnostic seven-day allowance and 23-day severity width. Read activation, decline and cancellation separately during case review: a fast cancellation is not a better customer outcome.
- The offer-response interval is usable for only 3,254 applications (24.9% of all in-scope applications). Another 8,072 have no observed offer-sent activation. An apparently strong offer-response result cannot describe the whole application population, and first endpoints may belong to different offers unless the offer identity is checked.
- December 2011 and January 2012 start cohorts are the first two investigation leads under each of the three baseline views, although their order changes. Keep all later events of those applications. The comparison does not establish a seasonal cause, staff responsibility or an approved service-level breach.

Begin with cases from those cohorts that combine an observable outcome interval with repeated document or offer work. Confirm whether the repeated work reflects missing information, an avoidable correction or customer choice before selecting a checklist or communication pilot.

In [ ]:
for name, run in scored_runs.items():
    display(Markdown(f"### {name}"))
    display(integrity(run))
display(journal(runs))

## 13. Process improvement and support: choose a feasible pilot

The proposals below are conditional on the case review. They change the business process, not the software package. Select one supported mechanism, a responsible owner and comparable cases. Agree the outcome and quality safeguard before starting. An improved WISE score alone is not a business benefit.

In [ ]:
process_improvements = pd.DataFrame([{'Process issue to confirm': 'Repeated requests for the same missing documents', 'Pilot change': 'Provide one consolidated document checklist and one contact owner at the initial assessment', 'Owner': 'Lending operations lead', 'Success measure': 'Shorter decision interval and fewer repeated document requests within comparable application outcomes and amount cohorts', 'Safeguard': 'No deterioration in required checks, applicant withdrawals or post-decision corrections'}, {'Process issue to confirm': 'Applicant delay caused by unclear offer terms or next steps', 'Pilot change': 'Explain required acceptance steps and give an agreed contact channel when the first offer is sent', 'Owner': 'Customer service lead', 'Success measure': 'Faster complete offer responses, with a separate record of customer-requested changes', 'Safeguard': 'Preserve customer choice; do not pressure customers into acceptance'}, {'Process issue to confirm': 'Internal offer corrections recur in the same cohort', 'Pilot change': 'Review the recurring correction reason and check completeness before offer creation', 'Owner': 'Offer administration lead', 'Success measure': 'Fewer avoidable corrections and stable application decision quality', 'Safeguard': 'Retain legitimate alternative offers and the necessary credit decision'}])
display(process_improvements)

## 14. Full analysis journal and next iteration

The execution journal records what was actually loaded and assessed. The decision journal records the remaining business review and pilot decisions. Replace the pending notes after a stakeholder review; they do not imply that an organisation has approved a norm or that a pilot has occurred. If a case changes the explanation, return to the relevant PM² stage and rerun the same complete-case assessment.

Saved tables and figures may contain case identifiers or business context. Export is optional and disabled initially; keep shared results consistent with the source’s permitted use.

In [ ]:
decision_journal = pd.DataFrame([
    {"PM² stage": "Planning", "Decision": "Confirm the business goal, acceptable outcomes and policy owner", "Evidence to record": "Owner-approved scope and diagnostic cut points", "Status": "Pending business review"},
    {"PM² stage": "Extraction", "Decision": "Confirm source identity, observation boundary and case unit", "Evidence to record": "Execution journal and source identifiers", "Status": "Review computed journal"},
    {"PM² stage": "Data processing", "Decision": "Retain whole cases and review changing context or unavailable endpoints", "Evidence to record": "Context profiles and full-case witnesses", "Status": "Review computed evidence"},
    {"PM² stage": "Mining and analysis", "Decision": "Choose a lead that survives meaningful preference and threshold scenarios", "Evidence to record": "Coverage, signed drivers and shortlist sensitivity", "Status": "Review computed comparisons"},
    {"PM² stage": "Evaluation", "Decision": "Distinguish avoidable work, necessary control and recording gaps", "Evidence to record": "Source-case explanation and integrity checks", "Status": "Pending case interpretation"},
    {"PM² stage": "Process improvement and support", "Decision": "Approve one feasible pilot with a business outcome and stopping rule", "Evidence to record": "Owner, comparable cases, quality safeguards and follow-up date", "Status": "No pilot undertaken"},
])
display(journal(runs))
display(decision_journal)
if EXPORT_RESULTS:
    save_outputs(runs, Path("results") / "BPIC2012")